# 第70章 地图图表（Map / Geo）

用Geo地图表达国家或地区的空间位置、规模和差异。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

地理位置本身对解释有意义，例如市场、网点或区域指标。

## 数据结构

标准地理编码、名称以及数值指标。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 projection="natural earth" 改为 "orthographic" 或 "mercator"，对比不同地图投影的效果
2. 修改 color_continuous_scale 从 "Blues" 为 "YlOrRd"，观察色盘对指标差异的表达
3. 添加 hover_data 显示增长率等补充字段，说明悬浮信息对空间数据解读的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


funnel = pd.DataFrame({
    "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
    "users": [12000, 7200, 3100, 1850, 1420],
})
timeline = pd.DataFrame({
    "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
    "start": pd.to_datetime(["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]),
    "finish": pd.to_datetime(["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]),
    "owner": ["数据", "分析", "分析", "负责人"],
})
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    date = pd.Timestamp("2026-01-01"), category=diamonds["cut"], region=diamonds["clarity"],
    channel = diamonds["color"], order_value=diamonds["price"], items=diamonds["carat"],
    sales = diamonds["price"], month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
monthly = flights.query("year == 1960").rename(columns={"passengers": "sales"}).copy()
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()
regional = orders_full.groupby(["region", "channel"], as_index=False)["sales"].sum()
hierarchy = diamonds.groupby(["cut", "color"], as_index=False)["price"].sum().rename(
    columns = {"cut": "department", "color": "category", "price": "sales"}
)
gapminder = pd.read_csv(f"{base_url}/datasets/gapminder.csv")
countries = gapminder.query("year == 2007").assign(
    country = lambda frame: frame["country"], market=lambda frame: frame["country"],
    sales = lambda frame: frame["gdpPercap"], growth=lambda frame: frame["lifeExp"]
)
print(f"Diamonds：{len(diamonds):,} 行；Flights：{len(flights):,} 行；Gapminder：{len(gapminder):,} 行")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.scatter_geo(countries, locations="country", locationmode="country names", size="sales", color="growth", hover_name="market", projection="natural earth", color_continuous_scale="Blues", title="Gapminder：人均GDP与预期寿命")
fig.update_layout(coloraxis_colorbar_title="预期寿命")
fig.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.choropleth(countries, locations="country", locationmode="country names", color="sales", hover_name="market", hover_data={"growth": ":.1f"}, projection="natural earth", color_continuous_scale="YlGnBu", title="Gapminder：各国人均GDP")
fig.update_layout(coloraxis_colorbar_title="人均GDP")
fig.show()


## 3. 参数说明

- locations：地理编码
- locationmode：编码类型
- projection：投影
- scope：区域范围


## 4. 结果解读

结合位置和指标读取空间模式；地图面积不能替代数值比较。


## 常见误区

- 地理编码无法匹配
- 大区域视觉面积造成偏见
- 无空间意义的数据强行使用地图


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
map_points = pd.DataFrame({
    "city": ["上海", "广州", "北京", "成都"],
    "lat": [31.23, 23.13, 39.90, 30.57],
    "lon": [121.47, 113.26, 116.40, 104.07],
    "sales": [320, 250, 280, 190],
})
fig = px.scatter_geo(map_points, lat="lat", lon="lon", size="sales", hover_name="city", projection="natural earth", title="国内城市销售点位")
fig.update_geos(lataxis_range=[15, 55], lonaxis_range=[70, 140])
fig.show()


## 本章小结

用Geo地图表达国家或地区的空间位置、规模和差异。


### 你已经掌握

- 判断地图图表（Map / Geo）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 地理位置本身对解释有意义，例如市场、网点或区域指标。 |
| 数据结构 | 标准地理编码、名称以及数值指标。 |
| 结果解读 | 结合位置和指标读取空间模式；地图面积不能替代数值比较。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `locations` | 地理编码 |
| `locationmode` | 编码类型 |
| `projection` | 投影 |
| `scope` | 区域范围 |


### 需要注意

- 地理编码无法匹配
- 大区域视觉面积造成偏见
- 无空间意义的数据强行使用地图


### 完成检查

- [ ] 能判断什么问题适合使用地图图表（Map / Geo）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
